# Notebook 03: Stylometric Feature Extraction & Decay Analysis
**Ship of Theseus — Computational Forensics Project**

Extract **5+ Stylometric Features** T0 → T3 and generate at least one decay plot.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path().resolve().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import DATA_PROCESSED, DATASETS, STAGES, FAMILIES

FIGURES = ROOT / "figures" / "stylometry"
EXP_DATA = ROOT / "experiments" / "stylometry"
FIGURES.mkdir(parents=True, exist_ok=True)
EXP_DATA.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

## Load Chain Data
Load T0 → T3 chains from `data/processed/`. Sample 500 rows per dataset for speed (spaCy parsing is slower than similarity metrics).

In [ ]:
SAMPLE_N = 500
SEED = 42

chains = {}
for name in DATASETS.keys():
    path = DATA_PROCESSED / f"{name}_chains.csv"
    df = pd.read_csv(path)
    chains[name] = df.sample(min(SAMPLE_N, len(df)), random_state=SEED)
    print(f"Loaded '{name}': {chains[name].shape}")

print(f"\nColumns: {chains['wp'].columns.tolist()}")

## Extract Stylometric Features (5+ Features)

For each stage (T0 → T3), extract:
1. **Sentence length** (mean, variance)
2. **Type-Token Ratio (TTR)** — lexical diversity
3. **Punctuation ratio**
4. **POS tag frequencies** (NOUN, VERB, ADJ, ADV, etc.)
5. **Dependency tree depth** (mean, max)

Uses spaCy `en_core_web_sm` for POS tagging and dependency parsing.

In [ ]:
from src.features.stylometry import extract_features_batch, aggregate_features_by_stage

# Use a combined sample from all datasets for the main analysis
# Focus on 3 representative datasets to keep runtime manageable
FOCUS_DATASETS = ["wp", "xsum", "yelp"]

all_stage_features = {}

for name in FOCUS_DATASETS:
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"{'='*60}")
    stage_feats, sampled = aggregate_features_by_stage(
        chains[name], stages=("T0", "T1", "T2", "T3"), sample_n=SAMPLE_N, seed=SEED
    )
    all_stage_features[name] = stage_feats

    # Save per-stage features
    for stage, feat_df in stage_feats.items():
        feat_df.to_csv(EXP_DATA / f"{name}_{stage}_features.csv", index=False)

print("\nFeature extraction complete!")

## Feature Summary Table by Stage
Mean values of each stylometric feature across T0 → T3, aggregated across all focus datasets.

In [ ]:
SCALAR_FEATURES = ["sent_len_mean", "sent_len_var", "ttr", "punct_ratio",
                    "dep_depth_mean", "dep_depth_max"]

# Build summary table: mean of each feature per stage (across all datasets)
summary_rows = []
for stage in STAGES:
    combined = pd.concat([all_stage_features[name][stage] for name in FOCUS_DATASETS],
                         ignore_index=True)
    row = {"stage": stage}
    for feat in SCALAR_FEATURES:
        row[feat] = combined[feat].mean()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("stage").round(4)
print("Mean Stylometric Features by Stage (across all focus datasets):")
print(summary_df.to_string())
summary_df.to_csv(EXP_DATA / "feature_summary_by_stage.csv")

---
## Figure 1: Stylometric Feature Decay Curves (T0 → T3)
Normalize each feature to its T0 value and plot the decay. This shows which "planks" are replaced fastest.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: Absolute feature values ---
ax = axes[0]
colors = {'sent_len_mean': '#1565C0', 'ttr': '#E53935', 'dep_depth_mean': '#2E7D32',
          'punct_ratio': '#FF8F00', 'sent_len_var': '#7B1FA2', 'dep_depth_max': '#00838F'}
markers = {'sent_len_mean': 'o', 'ttr': 's', 'dep_depth_mean': '^',
           'punct_ratio': 'D', 'sent_len_var': 'v', 'dep_depth_max': 'P'}

for feat in SCALAR_FEATURES:
    vals = [summary_df.loc[s, feat] for s in STAGES]
    ax.plot(STAGES, vals, marker=markers[feat], color=colors[feat],
            linewidth=2, markersize=7, label=feat)
ax.set_title("Stylometric Features Across Stages", fontweight='bold')
ax.set_xlabel("Paraphrase Stage")
ax.set_ylabel("Feature Value")
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.3)

# --- Right panel: Normalized to T0 = 1.0 ---
ax = axes[1]
for feat in SCALAR_FEATURES:
    t0_val = summary_df.loc["T0", feat]
    if t0_val == 0:
        continue
    normed = [summary_df.loc[s, feat] / t0_val for s in STAGES]
    ax.plot(STAGES, normed, marker=markers[feat], color=colors[feat],
            linewidth=2, markersize=7, label=feat)
ax.axhline(1.0, color='grey', linestyle='--', alpha=0.5)
ax.set_title("Normalized Feature Decay (T0 = 1.0)", fontweight='bold')
ax.set_xlabel("Paraphrase Stage")
ax.set_ylabel("Relative to T0")
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / "fig1_feature_decay_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig1_feature_decay_curves.png")

## Figure 2: POS Tag Frequency Heatmap (T0 vs T3)
Compare how part-of-speech distributions shift from the original human text (T0) to the 3rd-generation paraphrase (T3).

In [ ]:
from src.features.stylometry import TRACKED_POS

pos_cols = [f"pos_{p}" for p in TRACKED_POS]
# Filter to POS tags that have meaningful frequency (> 0.01 on average)
MAIN_POS = ["NOUN", "VERB", "ADJ", "ADV", "PRON", "DET", "ADP", "PUNCT"]
main_pos_cols = [f"pos_{p}" for p in MAIN_POS]

# Build heatmap data: rows = POS tags, columns = stages
heatmap_data = {}
for stage in STAGES:
    combined = pd.concat([all_stage_features[name][stage] for name in FOCUS_DATASETS],
                         ignore_index=True)
    heatmap_data[stage] = combined[main_pos_cols].mean()

heatmap_df = pd.DataFrame(heatmap_data).round(4)
heatmap_df.index = MAIN_POS

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Absolute POS frequencies
ax = axes[0]
sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Frequency'})
ax.set_title("POS Tag Frequencies by Stage", fontweight='bold')
ax.set_ylabel("POS Tag")
ax.set_xlabel("Paraphrase Stage")

# Right: Change from T0 (delta)
ax = axes[1]
delta_df = heatmap_df.subtract(heatmap_df["T0"], axis=0)
sns.heatmap(delta_df, annot=True, fmt=".4f", cmap="RdBu_r", center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Change from T0'})
ax.set_title("POS Frequency Change from T0", fontweight='bold')
ax.set_ylabel("POS Tag")
ax.set_xlabel("Paraphrase Stage")

plt.tight_layout()
plt.savefig(FIGURES / "fig2_pos_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig2_pos_heatmap.png")

## Figure 3: Dependency Depth Distribution (T0 vs T3)
Box plots showing how syntactic complexity (dependency tree depth) changes across paraphrase iterations.

In [ ]:
# Build long-form dataframe for boxplots
dep_rows = []
for stage in STAGES:
    combined = pd.concat([all_stage_features[name][stage] for name in FOCUS_DATASETS],
                         ignore_index=True)
    for _, row in combined.iterrows():
        dep_rows.append({"stage": stage, "dep_depth_mean": row["dep_depth_mean"],
                         "dep_depth_max": row["dep_depth_max"],
                         "sent_len_var": row["sent_len_var"],
                         "ttr": row["ttr"]})
dep_long = pd.DataFrame(dep_rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Dependency depth (mean)
sns.boxplot(data=dep_long, x="stage", y="dep_depth_mean", ax=axes[0],
            palette="Blues", order=STAGES)
axes[0].set_title("Mean Dependency Depth by Stage", fontweight='bold')
axes[0].set_xlabel("Paraphrase Stage")
axes[0].set_ylabel("Mean Dependency Depth")

# TTR
sns.boxplot(data=dep_long, x="stage", y="ttr", ax=axes[1],
            palette="Reds", order=STAGES)
axes[1].set_title("Type-Token Ratio by Stage", fontweight='bold')
axes[1].set_xlabel("Paraphrase Stage")
axes[1].set_ylabel("TTR (Lexical Diversity)")

# Sentence length variance
sns.boxplot(data=dep_long, x="stage", y="sent_len_var", ax=axes[2],
            palette="Greens", order=STAGES)
axes[2].set_title("Sentence Length Variance by Stage", fontweight='bold')
axes[2].set_xlabel("Paraphrase Stage")
axes[2].set_ylabel("Variance (words)")

plt.tight_layout()
plt.savefig(FIGURES / "fig3_dep_depth_boxplots.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig3_dep_depth_boxplots.png")

## Figure 4: POS Distribution Jensen-Shannon Divergence (T0 vs Tn)
Quantifies how far the POS distribution at each stage has drifted from the original T0.

In [ ]:
from src.features.stylometry import compute_pos_divergence

fig, ax = plt.subplots(figsize=(8, 5))

for name in FOCUS_DATASETS:
    # Get mean POS distribution per stage
    t0_mean = all_stage_features[name]["T0"][main_pos_cols].mean().to_dict()
    js_vals = [0.0]  # T0 vs T0 = 0
    for stage in ["T1", "T2", "T3"]:
        tn_mean = all_stage_features[name][stage][main_pos_cols].mean().to_dict()
        js_vals.append(compute_pos_divergence(t0_mean, tn_mean))

    ax.plot(STAGES, js_vals, marker='o', linewidth=2, markersize=8, label=name)

ax.set_title("POS Distribution Drift from T0\n(Jensen-Shannon Divergence)", fontweight='bold')
ax.set_xlabel("Paraphrase Stage")
ax.set_ylabel("JS Divergence (0=identical, 1=max)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / "fig4_pos_js_divergence.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig4_pos_js_divergence.png")